# Parse HTML Docs
This notebook contains functions that:
1) Retrieves a list of fanwork links from a text file created with the functions in get_fanwork_links.ipynb
2) For each fanwork link, creates a BeautifulSoup object, parses the resulting html document, and extracts desired metadata about each fanwork
3) Saves the extracted metadata in a json structure

In [164]:
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver import ActionChains
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## TODO - remove once you've tested selenium
import requests

from bs4 import BeautifulSoup
import re
import json

# Retrieve fanworks list from text file

In [165]:
def get_fanwork_links(filepath):
    """Takes in a txt file of pagination links and returns a list."""
    fanwork_links = open(filepath).readlines()
    for i in range(len(fanwork_links)):
        fanwork_links[i] = fanwork_links[i].replace("\n","")
    return fanwork_links

# Create soup objects

In [166]:
def get_fanwork_html(url):
    driver = webdriver.Firefox()
    driver.get(url)

    # detect TOS agreement
    agreement_class_present = len(driver.find_elements(By.CLASS_NAME, value='agreement')) > 0
    print("TOS agreement section is present: ", str(agreement_class_present))

    # if agreement class is present, find TOS agreement checkboxes / submit button
    if agreement_class_present:
        wait = WebDriverWait(driver, 10)
        tos_agree = wait.until(EC.element_to_be_clickable((By.ID, 'tos_agree')))

        # click checkboxes + accept TOS
        actions = ActionChains(driver)
        actions.move_to_element(tos_agree).perform()
        actions.click(tos_agree).perform()

        data_processing_agree = wait.until(EC.element_to_be_clickable((By.ID, 'data_processing_agree')))
        actions.move_to_element(data_processing_agree).perform()
        actions.click(data_processing_agree).perform()

        accept_tos_button = wait.until(EC.element_to_be_clickable((By.ID, 'accept_tos')))
        actions.move_to_element(accept_tos_button).perform()
        actions.click(accept_tos_button).perform()

    else:
        pass

    # detect adult content warning
    accept_adult_content_present = len(driver.find_elements(By.LINK_TEXT, 'Yes, Continue')) > 0
    print("Adult content warning is present: ", str(accept_adult_content_present))

    if accept_adult_content_present:
        wait = WebDriverWait(driver, 15)
        accept_adult_content_button = wait.until(EC.element_to_be_clickable((By.LINK_TEXT, 'Yes, Continue')))

        # click link + accept adult content warning
        actions = ActionChains(driver)
        actions.move_to_element(accept_adult_content_button).perform()
        actions.click(accept_adult_content_button).perform()

    else:
        pass

    # return page source / html doc
    driver.implicitly_wait(10) # seconds

    # get OuterHTML from 'work meta group'
    work_meta_group = driver.find_element(By.CLASS_NAME, 'wrapper')
    outer_html = work_meta_group.get_attribute('outerHTML')

    return outer_html

In [409]:
def get_fanwork_html1(url):
    driver = webdriver.Firefox()
    driver.get(url)

    # detect TOS agreement
    agreement_class_present = len(driver.find_elements(By.CLASS_NAME, value='agreement')) > 0
    print("TOS agreement section is present: ", str(agreement_class_present))

    # if agreement class is present, find TOS agreement checkboxes / submit button
    if agreement_class_present:
        wait = WebDriverWait(driver, 10)
        tos_agree = wait.until(EC.element_to_be_clickable((By.ID, 'tos_agree')))

        # click checkboxes + accept TOS
        actions = ActionChains(driver)
        actions.move_to_element(tos_agree).perform()
        actions.click(tos_agree).perform()

        data_processing_agree = wait.until(EC.element_to_be_clickable((By.ID, 'data_processing_agree')))
        actions.move_to_element(data_processing_agree).perform()
        actions.click(data_processing_agree).perform()

        accept_tos_button = wait.until(EC.element_to_be_clickable((By.ID, 'accept_tos')))
        actions.move_to_element(accept_tos_button).perform()
        actions.click(accept_tos_button).perform()

    else:
        pass

    # detect adult content warning
    accept_adult_content_present = len(driver.find_elements(By.LINK_TEXT, 'Yes, Continue')) > 0
    print("Adult content warning is present: ", str(accept_adult_content_present))

    if accept_adult_content_present:
        wait = WebDriverWait(driver, 15)
        accept_adult_content_button = wait.until(EC.element_to_be_clickable((By.LINK_TEXT, 'Yes, Continue')))

        # click link + accept adult content warning
        actions = ActionChains(driver)
        actions.move_to_element(accept_adult_content_button).perform()
        actions.click(accept_adult_content_button).perform()

    else:
        pass

    # return page source / html doc
    driver.implicitly_wait(10) # seconds

    # get OuterHTML from 'work meta group'
    work_meta_group = driver.find_element(By.CLASS_NAME, 'work')
    inner_html = work_meta_group.get_attribute('innerHTML')

    return inner_html

In [ ]:
## TODO - delete once you've tested the other one
def get_html_doc_fanworks(url):
    """Takes in the url for any given Ao3 fanwork and returns an html document of that page."""
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

In [167]:
def get_soup(html):
    soup = BeautifulSoup(html, features='html.parser')
    return soup

In [168]:
def get_all_dt(soup):
    """Takes in BeautifulSoup object from Ao3 fanwork and returns a list of all 'dt' elements of the html document. 'dt' elements represent the metadata categories present for the fanwork."""
    all_dt = soup.find_all('dt')
    # all_dt = list(all_dt)
    return all_dt

In [169]:
def update_all_dt(dt_list):
    dt_dict = {
        '<dt><label for="user_session_login_small">Username or email:</label></dt>':'user_session_login',
        '<dt><label for="user_session_password_small">Password:</label></dt>':'user_session_password',
        '<dt class="rating tags">':'Rating',
        '<dt class="warning tags">':'Archive Warning',
        '<dt class="category tags">':'Categories',
        '<dt class="fandom tags">':'Fandoms',
        '<dt class="relationship tags">':'Relationships',
        '<dt class="character tags">':'Characters',
        '<dt class="freeform tags">':'Additional Tags',
        '<dt class="language">':'Language',
        '<dt class="stats">':'Stats',
        '<dt class="published">':'Date Published',
        '<dt class="status">':'Date Updated',
        '<dt class="words">':'Word Count',
        '<dt class="chapters">':'Chapters',
        '<dt class="comments">':'Comments',
        '<dt class="kudos">':'Kudos',
        '<dt class="bookmarks">':'Bookmarks',
        '<dt class="hits">':'Hits',
        '<dt class="landmark">Note:</dt>':'note',
        '<dt><label for="comment_name_for_211688701">Guest name</label></dt>':'guest_commenter_name',
        '<dt><label for="comment_email_for_211688701">Guest email</label></dt>':'guest_commenter_email'
    }

    for i in range(len(dt_list)):
        dt_list[i] = str(dt_list[i])

    for key in dt_dict.keys():
        for i in range(len(dt_list)):
            key_string = str(key)
            if key_string in dt_list[i]:
                dt_list[i] = dt_dict[key]
            else:
                continue
    return dt_list

In [170]:
def get_all_dd(soup):
    """Takes in BeautifulSoup object from Aoe fanwork and returns a list of all 'dd' elements of the html document. 'dd' elements represent the metadata values present in the fanwork."""
    all_dd = soup.find_all('dd')
    # all_dd = list(all_dd)

    # convert dd_list items to strings
    for i in range(len(all_dd)):
        all_dd[i] = str(all_dd[i])

    return all_dd

# Parse html docs and clean metadata

## strip_dd

In [171]:
def strip_dd(dd_list):
    text_to_strip = ['\n',
                 '<ul class="commas">',
                 '<li>','</li>',
                 '</ul>',
                 '<a class="tag" href="','</a>',
                 '<dd class="rating tags">',
                 '/tags/Not%20Rated/works">',
                 '/tags/General%20Audiences/works">',
                 '/tags/Teen%20And%20Up%20Audiences/works">',
                 '/tags/Mature/works">',
                 '/tags/Explicit/works">',
                 '<dd class="warning tags">',
                 '/tags/No%20Archive%20Warnings%20Apply/works">',
                 '/tags/Graphic%20Depictions%20Of%20Violence/works">',
                 '/tags/Choose%20Not%20To%20Use%20Archive%20Warnings/works">',
                 '/tags/Major%20Character%20Death/works">',
                 '/tags/Rape*s*Non-Con/works">',
                 '/tags/Underage%20Sex/works">',
                 '<dd class="category tags">',
                 '/tags/Gen/works">',
                 '/tags/F*s*F/works">',
                 '/tags/F*s*M/works">F/M',
                 '/tags/M*s*M/works">M/M',
                 '/tags/Multi/works">Multi',
                 '/tags/Other/works">Other',
                 '</dd>',
                 '<dd class="language" lang="en">',
                 '<!-- end of cache -->',
                 '<dd class="stats">',
                 '<dl class="stats">',
                 '<dt class="published">',
                 '</dt><dd class="published">',
                 '<dt class="status">',
                 '</dt><dd class="status">',
                 '<dt class="words">',
                 '</dt><dd class="words">',
                 '<dt class="chapters">',
                 '</dt><dd class="chapters">',
                 '<dt class="comments">',
                 '</dt><dd class="comments">',
                 '<dt class="kudos">',
                 '</dt><dd class="kudos">',
                 '<dt class="bookmarks">',
                 '</dt><dd class="bookmarks">',
                 'bookmarks">',
                 '<a href="/',
                 '<dt class="hits">',
                 '</dt><dd class="hits">',
                 '</dl>',
                 '<dd class="instructions comment_form">',
                 '<dd><input id="',
                 '" name="comment[name]" type="text"/><script>',
                 '/<![CDATA[var ',
                 ' = new LiveValidation',
                 ', { wait: 500, onlyOnBlur: false }',
                 '.add(Validate.Presence, {"failureMessage":"Please enter your name.","validMessage":""})',
                 ';//]]>',
                 '</script>',
                 '<input id="',
                 '" name="comment[email]"',
                 ' type="text"/>',
                 '<script>',
                 '.add(Validate.Presence, {"failureMessage":"Please enter your email address.","validMessage":""})',
                 '<dd><input autocomplete="on" ',
                 'id="user_session_login_small" '
                 ]
    for item in text_to_strip:
        for i in range(len(dd_list)):
            if item in dd_list[i]:
                dd_list[i] = dd_list[i].replace(item, "")
            else:
                continue
    return dd_list

## cleanup_fandom_tags

In [172]:
def cleanup_fandom_tags(dd_list):
    for i in range(len(dd_list)):
        if '<dd class="fandom tags">' in dd_list[i]:
            dd_list[i] = dd_list[i].replace('&amp;','&')
            dd_list[i] = dd_list[i].replace('*a*','&')
            dd_list[i] = dd_list[i].replace('%20','')
            dd_list[i] = dd_list[i].replace('<dd class="fandom tags">','')

            dd_list[i] = dd_list[i].split('/tags')

            pattern = re.compile(r"(?<=>)(?!.*>).*")
            for index in range(len(dd_list[i])):
                match = re.search(pattern, dd_list[i][index])
                if match:
                    dd_list[i][index] = match.group(0)
                else:
                    dd_list[i][index] = dd_list[i][index]

        else:
            continue
    return dd_list

## cleanup_relationship_tags

In [173]:
def cleanup_relationship_tags(dd_list):
    for i in range(len(dd_list)):
        if '<dd class="relationship tags">' in dd_list[i]:
            dd_list[i] = dd_list[i].replace('&amp;','&')
            dd_list[i] = dd_list[i].replace('*a*','&')
            dd_list[i] = dd_list[i].replace('%20','')
            dd_list[i] = dd_list[i].replace('<dd class="relationship tags">','')

            dd_list[i] = dd_list[i].split('/tags')

            pattern = re.compile(r"(?<=>)(?!.*>).*")
            for index in range(len(dd_list[i])):
                match = re.search(pattern, dd_list[i][index])
                if match:
                    dd_list[i][index] = match.group(0)
                else:
                    dd_list[i][index] = dd_list[i][index]

        else:
            continue
    return dd_list

## cleanup_character_tags

In [174]:
def cleanup_character_tags(dd_list):
    for i in range(len(dd_list)):
        if '<dd class="character tags">' in dd_list[i]:
            dd_list[i] = dd_list[i].replace('&amp;','&')
            dd_list[i] = dd_list[i].replace('*a*','&')
            dd_list[i] = dd_list[i].replace('%20','')
            dd_list[i] = dd_list[i].replace('<dd class="character tags">','')

            dd_list[i] = dd_list[i].split('/tags')

            pattern = re.compile(r"(?<=>)(?!.*>).*")
            for index in range(len(dd_list[i])):
                match = re.search(pattern, dd_list[i][index])
                if match:
                    dd_list[i][index] = match.group(0)
                else:
                    dd_list[i][index] = dd_list[i][index]

        else:
            continue
    return dd_list

## cleanup_additional_tags

In [175]:
def cleanup_additional_tags(dd_list):
    for i in range(len(dd_list)):
        if '<dd class="freeform tags">' in dd_list[i]:
            dd_list[i] = dd_list[i].replace('&amp;','&')
            dd_list[i] = dd_list[i].replace('*a*','&')
            dd_list[i] = dd_list[i].replace('%20','')
            dd_list[i] = dd_list[i].replace('<dd class="freeform tags">','')

            dd_list[i] = dd_list[i].split('/tags')

            pattern = re.compile(r"(?<=>)(?!.*>).*")
            for index in range(len(dd_list[i])):
                match = re.search(pattern, dd_list[i][index])
                if match:
                    dd_list[i][index] = match.group(0)
                else:
                    dd_list[i][index] = dd_list[i][index]

        else:
            continue
    return dd_list

# Retrieve Metadata

In [176]:
def get_rating(dt_list, dd_list):
    # find ratings index
    if 'Rating' in dt_list:
        index = dt_list.index('Rating')
        rating = dd_list[index]
        return rating

    else:
        rating = 'Unknown'
        return rating

In [177]:
def get_warnings(dt_list, dd_list):
    # find warning index
    if 'Archive Warning' in dt_list:
        index = dt_list.index('Archive Warning')
        warning = dd_list[index]
        return warning
    else:
        warning = 'No warnings listed'
        return warning

In [178]:
def get_category(dt_list, dd_list):
    # find category index
    if 'Categories' in dt_list:
        index = dt_list.index('Categories')
        category = dd_list[index]
        return category
    else:
        category = 'Unknown'
        return category

In [179]:
def get_language(dt_list, dd_list):
    """Uses language index found in find_language_index to get language metadata value for an Ao3 work."""
    if 'Language' in dt_list:
        index = dt_list.index("Language")
        language_str = dd_list[index]
        language_str = language_str.strip()
        return language_str
    else:
        language_str = 'Unknown'
        return language_str

In [180]:
def get_fandom(dt_list, dd_list):
    # find fandom index
    if 'Fandoms' in dt_list:
        index = dt_list.index("Fandoms")
        fandoms = dd_list[index]
        return fandoms
    else:
        fandoms = 'No fandoms listed'
        return fandoms

In [181]:
def get_relationships(dt_list, dd_list):
    # find relationship index
    if 'Relationships' in dt_list:
        index = dt_list.index("Relationships")
        relationships = dd_list[index]
        return relationships
    else:
        relationships = 'No relationships listed'
        return relationships

In [182]:
def get_characters(dt_list, dd_list):
    # find relationship index
    if 'Characters' in dt_list:
        index = dt_list.index('Characters')
        characters = dd_list[index]
        return characters
    else:
        characters = 'No characters listed'
        return characters

In [183]:
def get_additional_tags(dt_list, dd_list):
    # find additional tags index
    if 'Additional Tags' in dt_list:
        index = dt_list.index("Additional Tags")
        additional_tags = dd_list[index]
        return additional_tags
    else:
        additional_tags = 'No additional tags'
        return additional_tags

In [184]:
def is_series(dt_list):
    for i in range(len(dt_list)):
        current_string = dt_list[i]
        if 'Series' in current_string:
            is_part_of_series = True
            return is_part_of_series
        else:
            is_part_of_series = False
            return is_part_of_series

In [185]:
def get_date_published(dt_list, dd_list):
    # find date published index
    if 'Date Published' in dt_list:
        index = dt_list.index('Date Published')
        date_published = dd_list[index]
        date_published = date_published.split('>')
        date_published = date_published[1]
        return date_published

    else:
        date_published = 'Unknown'
        return date_published

In [186]:
def get_date_updated(dt_list, dd_list):
    # find date updated index
    if 'Date Updated' in dt_list:
        index = dt_list.index("Date Updated")
        date_updated = dd_list[index]
        date_updated = date_updated.split('>')
        date_updated = date_updated[1]
        return date_updated
    elif 'Date Published' in dt_list:
        index = dt_list.index('Date Published')
        date_updated = dd_list[index]
        date_updated = date_updated.split('>')
        date_updated = date_updated[1]
        return date_updated
    else:
        date_updated = 'Unknown'
        return date_updated

In [187]:
def get_word_count(dt_list, dd_list):
    # find word count index
    if 'Word Count' in dt_list:
        index = dt_list.index("Word Count")
        word_count = dd_list[index]
        word_count = word_count.replace(',','')
        word_count = word_count.split('>')
        word_count = int(word_count[1])
        return word_count
    else:
        word_count = 'Unknown'
        return word_count

In [188]:
def get_chapter_count(dt_list, dd_list):
    # find Chapter index
    if 'Chapters' in dt_list:
        index = dt_list.index("Chapters")
        chapter_count = dd_list[index]
        chapter_count = chapter_count.split('>')
        chapter_count = chapter_count[1]
        return chapter_count
    else:
        chapter_count = str(1)
        return chapter_count

In [189]:
def get_comment_count(dt_list, dd_list):
    # find comment count index
    if 'Comments' in dt_list:
        index = dt_list.index("Comments")
        comment_count = dd_list[index]
        comment_count = comment_count.split('>')
        comment_count = int(comment_count[1])
        return comment_count
    else:
        comment_count = 0
        return comment_count

In [190]:
def get_kudos_count(dt_list, dd_list):
    # find kudos count index
    if 'Kudos' in dt_list:
        index = dt_list.index("Kudos")
        kudos_count = dd_list[index]
        kudos_count = kudos_count.split('>')
        kudos_count = int(kudos_count[1])
        return kudos_count
    else:
        kudos_count = 0
        return kudos_count

In [191]:
def get_bookmarks_count(dt_list, dd_list):
    # find bookmarks count index
    if 'Bookmarks' in dt_list:
        index = dt_list.index("Bookmarks")
        bookmarks_count = dd_list[index]
        bookmarks_count = bookmarks_count[-1]
        bookmarks_count = int(bookmarks_count)
        return bookmarks_count

    else:
        bookmarks_count = 0
        return bookmarks_count

# Create Dictionary from html items

In [192]:
def create_fanwork_dict(dt_list, dd_list):
    fanwork_dict = {}

    # add rating
    rating = get_rating(dt_list, dd_list)
    fanwork_dict['Rating'] = get_rating(dt_list, dd_list)

    # add archive warning(s)
    fanwork_dict['Archive Warnings'] = get_warnings(dt_list, dd_list)

    # category
    fanwork_dict['Category'] = get_category(dt_list, dd_list)

    # add fandom
    fanwork_dict['Fandom'] = get_fandom(dt_list, dd_list)

    # add characters
    fanwork_dict['Characters'] = get_characters(dt_list, dd_list)

    # add relationships
    fanwork_dict['Relationships'] = get_relationships(dt_list, dd_list)

    # add additional tags
    fanwork_dict['Additional Tags'] = get_additional_tags(dt_list, dd_list)

    # add language
    fanwork_dict['Language'] = get_language(dt_list, dd_list)

    # add whether it is a series
    fanwork_dict['Is Series'] = is_series(dd_list)

    # add stats
    fanwork_dict['Date Published'] = get_date_published(dt_list, dd_list)
    fanwork_dict['Date Updated'] = get_date_updated(dt_list, dd_list)
    fanwork_dict['Word Count'] = get_word_count(dt_list, dd_list)
    fanwork_dict['Chapter Count'] = get_chapter_count(dt_list, dd_list)
    fanwork_dict['Comment Count'] = get_comment_count(dt_list, dd_list)
    fanwork_dict['Kudos Count'] = get_kudos_count(dt_list, dd_list)
    fanwork_dict['Bookmarks Count'] = get_bookmarks_count(dt_list, dd_list)

    return fanwork_dict

# Add fanworks_dict to master dictionary (all_works_dict)

In [193]:
def create_all_works_dict():
    dict_all_works = {}
    return dict_all_works

In [194]:
def append_all_works_dict(dict_fanwork, dict_all_works):
    current_length = len(dict_all_works)
    new_idx = current_length + 1
    dict_all_works[new_idx] = dict_fanwork
    return dict_all_works

# Convert master dictionary to json object

In [195]:
def convert_to_json(dict_all_works):
    json_string = json.dumps(dict_all_works, indent=4)
    return json_string

In [196]:
def save_json_to_file(json_string):
    with open('data/json_string.json', "w") as f:
        f.write(json_string)

# Testing

In [ ]:
# test get_fanwork_links
fanwork_links = get_fanwork_links('../../data/fanwork_links.txt')
print(fanwork_links)

In [ ]:
print(len(fanwork_links))

## Individual link

In [410]:
# testing with getwhole work version
test_url1 = 'https://archiveofourown.org/works/51010177/chapters/128876734'
work_meta_group = get_fanwork_html1(test_url1)
print(work_meta_group)

TOS agreement section is present:  True
Adult content warning is present:  True

  <li id="work_51010177" class="work blurb group work-51010177 user-19010446" role="article">
    <!--title, author, fandom-->
  <div class="header module">
    <!-- updated_at=1773017854 -->
    <h4 class="heading">
      <a href="/works/51010177">i'll be the north star that takes you home</a>
      by

      <!-- do not cache -->
      <a rel="author" href="/users/heckblade/pseuds/heckblade">heckblade</a>



        for <a href="/users/ManofMud/gifts">ManofMud</a>
      
      
    </h4>

    <h5 class="fandoms heading">
      <span class="landmark">Fandoms:</span>
      <a class="tag" href="/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works">Candela Obscura (Critical Role Web Series)</a>
      &nbsp;
    </h5>

    <!--required tags-->
    <ul class="required-tags">
<li><a class="help symbol question modal modal-attached" title="Symbols key" href="/help/symbols_key" aria-controls="modal"><

In [197]:
# test get_html_doc_fanworks
test_url = 'https://archiveofourown.org/works/73925776'
outer_html = get_fanwork_html(test_url)
print(outer_html)

TOS agreement section is present:  True
Adult content warning is present:  True
<div id="outer" class="wrapper">
      <ul id="skiplinks"><li><a href="#main">Main Content</a></li></ul>
      <noscript><p id="javascript-warning">While we&#39;ve done our best to make the core functionality of this site accessible without JavaScript, it will work better with it enabled. Please consider turning it on!</p></noscript>

<!-- BEGIN header -->

<header id="header" class="region">

  <h1 class="heading">
    <a href="/"><span>Archive of Our Own</span><img alt="Archive of Our Own" class="logo" src="/images/ao3_logos/logo_42.png"></a>
  </h1>

    <div id="login" class="dropdown" aria-haspopup="true">
      <p class="user actions">
        <a id="login-dropdown" href="/users/login?return_to=%2Fworks%2F73925776" class="dropdown-toggle" data-toggle="dropdown" data-target="#">Log In</a>
      </p>
      <div id="small_login" class="simple login">
	<form class="new_user" id="new_user_session_small" ac

In [203]:
soup = get_soup(outer_html)

In [322]:
print(soup)

<div class="wrapper" id="outer">
<ul id="skiplinks"><li><a href="#main">Main Content</a></li></ul>
<noscript><p id="javascript-warning">While we've done our best to make the core functionality of this site accessible without JavaScript, it will work better with it enabled. Please consider turning it on!</p></noscript>
<!-- BEGIN header -->
<header class="region" id="header">
<h1 class="heading">
<a href="/"><span>Archive of Our Own</span><img alt="Archive of Our Own" class="logo" src="/images/ao3_logos/logo_42.png"/></a>
</h1>
<div aria-haspopup="true" class="dropdown" id="login">
<p class="user actions">
<a class="dropdown-toggle" data-target="#" data-toggle="dropdown" href="/users/login?return_to=%2Fworks%2F73925776" id="login-dropdown">Log In</a>
</p>
<div class="simple login" id="small_login">
<form accept-charset="UTF-8" action="/users/login?return_to=%2Fworks%2F73925776" class="new_user" id="new_user_session_small" method="post"><input autocomplete="off" name="authenticity_token"

In [204]:
warning_class = soup.find(attrs={'class':'warnings'})

In [205]:
print(warning_class)

<span class="warning-no warnings" title="No Archive Warnings Apply"><span class="text">No Archive Warnings Apply</span></span>


In [ ]:
print(type(warning_class))
print(len(warning_class))

In [207]:
warning_classes = soup.find_all(attrs={'class':'warnings'})

In [208]:
print(warning_classes)
print(len(warning_classes))

[<span class="warning-no warnings" title="No Archive Warnings Apply"><span class="text">No Archive Warnings Apply</span></span>, <li class="warnings"><strong><a class="tag" href="/tags/No%20Archive%20Warnings%20Apply/works">No Archive Warnings Apply</a></strong></li>]
2


In [211]:
warnings = []
for item in warning_classes:
    tag = item
    string = tag.string
    warnings.append(string)
    print(string)

No Archive Warnings Apply
No Archive Warnings Apply


In [216]:
warnings1 = list(set(warnings))
print(warnings1)

['No Archive Warnings Apply']


In [393]:
def get_warnings1(soup):
    warning_class = soup.find_all(attrs={'class':'warnings'})
    if len(warning_class) == 0:
        warning_list = "Warnings not tagged"
    else:
        warnings_list = []
        for item in warning_class:
            tag = item
            string = tag.string
            warnings_list.append(string)
        warnings_list = list(set(warnings_list))
        for i in range(len(warnings_list)):
            warnings_list[i] = str(warnings_list[i])
        if len(warnings_list) == 1:
            warnings_list = warnings_list[0]
        else:
            pass
    return warnings_list

In [394]:
warning_test = get_warnings1(soup)

In [395]:
print(warning_test)

No Archive Warnings Apply


In [391]:
def get_rating1(soup):
    rating_class = soup.find_all(attrs={'class':'rating'})
    if len(rating_class) == 0:
        rating_list = "Rating not tagged"
    else:
        rating_list = []
        for item in rating_class:
            tag = item
            string = tag.string
            rating_list.append(string)
        rating_list = list(set(rating_list))
        for i in range(len(rating_list)):
            rating_list[i] = str(rating_list[i])
        if len(rating_list) == 1:
            rating_list = rating_list[0]
        else:
            pass
    return rating_list

In [392]:
rating_test = get_rating1(soup)
print(rating_test)

Explicit


In [389]:
def get_category1(soup):
    category_class = soup.find_all(attrs={'class':'category'})
    if len(category_class) == 0:
        category_list = "No categories tagged"
    else:
        category_list = []
        for item in category_class:
            tag = item
            string = tag.string
            category_list.append(string)
        rating_list = list(set(category_list))
        for i in range(len(category_list)):
            category_list[i] = str(category_list[i])
        if len(category_list) == 1:
            category_list = category_list[0]
        else:
            pass
    return category_list

In [390]:
category_test = get_category1(soup)
print(category_test)

M/M


In [257]:
fandom_class = soup.find_all(attrs={'class':'fandoms heading'})

In [387]:
def get_relationships1(soup):
    relationships_class = soup.find_all(attrs={'class':'relationships'})
    if len(relationships_class) == 0:
        relationships_list = 'No relationships tagged'
    else:
        relationships_list = []
        for item in relationships_class:
            string = str(item.string)
            relationships_list.append(string)
        if len(relationships_list) == 1:
            relationships_list = relationships_list[0]
        else:
            pass
    return relationships_list

In [388]:
relationships_test2 = get_relationships1(soup)
print(relationships_test2)
print(type(relationships_test2))

Leo Amicus/Original Character(s)
<class 'str'>


In [385]:
def get_characters1(soup):
    character_class = soup.find_all(attrs={'class':'characters'})
    if len(character_class) == 0:
        character_list = 'No characters tagged'
    else:
        character_list = []
        for item in character_class:
            string = str(item.string)
            character_list.append(string)
    return character_list

In [386]:
character_test = get_characters1(soup)
print(character_test)

['Leo Amicus', 'Original Male Character(s)']


In [383]:
def get_additional_tags1(soup):
    freeform_class = soup.find_all(attrs={'class':'freeforms'})
    if len(freeform_class) == 0:
        freeform_list = 'No additional tags'
    else:
        freeform_list = []
        for item in freeform_class:
            string = str(item.string)
            freeform_list.append(string)
    return freeform_list

In [414]:
def get_date_published1(soup):
    published_class = soup.find_all(attrs={'class':'published'})
    if len(published_class) == 0:
        published_list = 'Unknown'
    else:
        published_list = []
        for item in published_class:
            string = str(item.string)
            published_list.append(string)
    return published_list

In [415]:
date_published_test = get_date_published1(soup)
print(date_published_test)

Unknown


In [398]:
date_published_class = soup.find_all(attrs={'class':'updated'})
print(date_published_class)

[]


In [411]:
date_updated_class1 = soup.find_all(attrs={'class':'datetime'})
print(date_updated_class1)

[<p class="datetime">09 Nov 2025</p>]


In [412]:
def get_date_updated1(soup):
    updated_class = soup.find_all(attrs={'class':'datetime'})
    if len(updated_class) == 0:
        updated_list = 'Unknown'
    else:
        updated_list = []
        for item in updated_class:
            string = str(item.string)
            updated_list.append(string)
    return updated_list

In [413]:
test_date_updated = get_date_updated1(soup)
print(test_date_updated)

['09 Nov 2025']


In [397]:
stats_class = soup.find_all(attrs={'class':'stats'})
print(stats_class)

[<dl class="stats">
<dt class="language">Language:</dt>
<dd class="language" lang="en">English</dd>
<dt class="words">Words:</dt>
<dd class="words">1,070</dd>
<dt class="chapters">Chapters:</dt>
<dd class="chapters">1/2</dd>
<dt class="comments">Comments:</dt>
<dd class="comments"><a href="/works/73925776?show_comments=true#comments">2</a></dd>
<dt class="kudos">Kudos:</dt>
<dd class="kudos"><a href="/works/73925776#kudos">2</a></dd>
<dt class="hits">Hits:</dt>
<dd class="hits">28</dd>
</dl>]


In [384]:
additional_tags_test = get_additional_tags1(soup)
print(additional_tags_test)

['Circle of the Crimson Mirror (Candela Obscura)', 'Boats and Ships', 'Crew of the Dandridge', 'Missing Scene', 'Blow Jobs', 'Anal Sex', 'Topping from the Bottom', 'Plot What Plot/Porn Without Plot', 'almost', 'Age Difference']


In [258]:
print(fandom_class)

[<h5 class="fandoms heading">
<span class="landmark">Fandoms:</span>
<a class="tag" href="/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works">Candela Obscura (Critical Role Web Series)</a>
       
    </h5>]


In [260]:
soup.find(attrs={'class':'fandoms heading'})

<h5 class="fandoms heading">
<span class="landmark">Fandoms:</span>
<a class="tag" href="/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works">Candela Obscura (Critical Role Web Series)</a>
       
    </h5>

In [262]:
fandom_test1 = soup.find(attrs={'class':'fandoms heading'}).find('a')

In [263]:
print(fandom_test1)
print(type(fandom_test1))

<a class="tag" href="/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works">Candela Obscura (Critical Role Web Series)</a>
<class 'bs4.element.Tag'>


In [253]:
fandom = soup.find('h5').find('a')
print(fandom)
print(type(fandom))

<a class="tag" href="/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works">Candela Obscura (Critical Role Web Series)</a>
<class 'bs4.element.Tag'>


In [265]:
print(fandom.string)
print(type(fandom.string))

Candela Obscura (Critical Role Web Series)
<class 'bs4.element.NavigableString'>


In [ ]:
fandom(str)

In [249]:
for child in fandom_class:
    print(child)

<h5 class="fandoms heading">
<span class="landmark">Fandoms:</span>
<a class="tag" href="/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works">Candela Obscura (Critical Role Web Series)</a>
       
    </h5>


In [261]:
print(type(fandom_class))

<class 'bs4.element.ResultSet'>


In [270]:
def get_fandom1(soup):
    fandom_class = soup.find(attrs={'class':'fandoms heading'}).find('a')
    fandom_list = []
    for item in fandom_class:
        string = str(item.string)
        fandom_list.append(string)
    fandom_list = list(set(fandom_list))
    if len(fandom_list) == 1:
        fandom_list = fandom_list[0]
    else:
        pass
    return fandom_list

In [271]:
fandom_test = get_fandom1(soup)
print(fandom_test)

Candela Obscura (Critical Role Web Series)


In [297]:
def get_language1(soup):
    language_class = soup.find_all(attrs={'class':'language'})
    language_list = []
    for item in language_class:
        tag = item
        string = str(tag.string)
        language_list.append(string)
    language = language_list[1]
    return (language)

In [298]:
language_test = get_language1(soup)
print(language_test)

English


In [327]:
def get_word_count1(soup):
    word_count_class = soup.find_all(attrs={'class':'words'})
    word_count_list = []
    for item in word_count_class:
        string = str(item.string)
        word_count_list.append(string)
    word_count = word_count_list[1]
    word_count = word_count.replace(",","")
    word_count = int(word_count)
    return word_count

In [329]:
word_count_test = get_word_count1(soup)c
print(word_count_test)
print(type(word_count_test))

1070
<class 'int'>


In [335]:
def get_chapter_count1(soup):
    chapter_count_class = soup.find_all(attrs={'class':'chapters'})
    chapter_count_list = []
    for item in chapter_count_class:
        string = str(item.string)
        chapter_count_list.append(string)
    chapter_count = chapter_count_list[1]
    return chapter_count

In [336]:
chapter_count_test = get_chapter_count1(soup)
print(chapter_count_test)

1/2


In [ ]:
def get_all_required_tags(soup):
    """Takes in BeautifulSoup object from Ao3 fanwork and returns a list of all 'dt' elements of the html document. 'dt' elements represent the metadata categories present for the fanwork."""
    all_dt = soup.find_all('dt')
    # all_dt = list(all_dt)
    return all_dt

In [272]:
# test get_all_dt
all_dt = get_all_dt(soup)
print(all_dt)

[<dt><label for="user_session_login_small">Username or email:</label></dt>, <dt><label for="user_session_password_small">Password:</label></dt>, <dt class="language">Language:</dt>, <dt class="words">Words:</dt>, <dt class="chapters">Chapters:</dt>, <dt class="comments">Comments:</dt>, <dt class="kudos">Kudos:</dt>, <dt class="hits">Hits:</dt>]


In [337]:
comment_class = soup.find_all(attrs={'class':'comments'})
print(comment_class)

[<dt class="comments">Comments:</dt>, <dd class="comments"><a href="/works/73925776?show_comments=true#comments">2</a></dd>]


In [381]:
def get_comment_count1(soup):
    comments_class = soup.find_all(attrs={'class':'comments'})
    if len(comments_class) == 0:
        comment_count = 0
    else:
        comment_count_list = []
        for item in comments_class:
            string = str(item.string)
            comment_count_list.append(string)
        comment_count = int(comment_count_list[1])
    return comment_count

In [382]:
test_comment_count = get_comment_count1(soup)
print(test_comment_count)
print(type(test_comment_count))

2
<class 'int'>


In [352]:
kudos_class = soup.find_all(attrs={'class':'kudos'})
print(kudos_class)

[<dt class="kudos">Kudos:</dt>, <dd class="kudos"><a href="/works/73925776#kudos">2</a></dd>]


In [379]:
def get_kudos_count1(soup):
    kudos_class = soup.find_all(attrs={'class':'kudos'})
    if len(kudos_class) == 0:
        kudos_count = 0
    else:
        kudos_count_list = []
        for item in kudos_class:
            string = str(item.string)
            kudos_count_list.append(string)
        kudos_count = int(kudos_count_list[1])
    return kudos_count

In [380]:
test_kudos_count = get_kudos_count1(soup)
print(test_kudos_count)
print(type(test_kudos_count))

2
<class 'int'>


In [367]:
bookmarks = soup.find_all(attrs={'class':'bookmarks'})
print(bookmarks)
print(len(bookmarks))

[]
0


In [374]:
def get_bookmarks_count1(soup):
    bookmarks_class = soup.find_all(attrs={'class':'bookmarks'})
    if len(bookmarks_class) == 0:
        bookmarks_count = 0
    else:
        bookmarks_count_list = []
        for item in bookmarks_class:
            string = str(item.string)
            bookmarks_count_list.append(string)
        bookmarks_count = int(bookmarks_count_list[1])
    return bookmarks_count

In [375]:
bookmarks_test = get_bookmarks_count1(soup)
print(bookmarks_test)
print(type(bookmarks_test))

0
<class 'int'>


In [368]:
def get_hits_count1(soup):
    hits_class = soup.find_all(attrs={'class':'hits'})
    if len(hits_class) == 0:
        hits_count = 0
    else:
        hits_count_list = []
        for item in hits_class:
            string = str(item.string)
            hits_count_list.append(string)
        hits_count = hits_count_list[1]
    return hits_count

In [369]:
test_hits_count = get_hits_count1(soup)
print(test_hits_count)

28


In [273]:
all_dt_updated = update_all_dt(all_dt)
print(all_dt_updated)

['user_session_login', 'user_session_password', 'Language', 'Word Count', 'Chapters', 'Comments', 'Kudos', 'Hits']


In [ ]:
print(len(all_dt))

In [278]:
# test get_all_dd
all_dd = get_all_dd(soup)

In [281]:
print(all_dd)

['name="user[login]"', 'user_session_password_small" name="user[password]" type="password"/>', 'English', '<dd class="words">1,070', '<dd class="chapters">1/2', '<dd class="comments">works/73925776?show_comments=true#comments">2', '<dd class="kudos">works/73925776#kudos">2', '<dd class="hits">28']


In [279]:
all_dd_stripped = strip_dd(all_dd)

In [280]:
print(all_dd_stripped)

['name="user[login]"', 'user_session_password_small" name="user[password]" type="password"/>', 'English', '<dd class="words">1,070', '<dd class="chapters">1/2', '<dd class="comments">works/73925776?show_comments=true#comments">2', '<dd class="kudos">works/73925776#kudos">2', '<dd class="hits">28']


In [ ]:
# test cleanup_fandom_tags
all_dd_stripped = cleanup_fandom_tags(all_dd_stripped)

In [ ]:
# test cleanup_relationship_tags
all_dd_stripped = cleanup_relationship_tags(all_dd_stripped)

In [ ]:
# test cleanup_character_tags
all_dd_stripped = cleanup_character_tags(all_dd_stripped)

In [ ]:
# test cleanup_additional_tags
all_dd_stripped = cleanup_additional_tags(all_dd_stripped)

In [ ]:
# test get_ratings
get_rating(all_dt_updated, all_dd_stripped)

In [ ]:
# test get_warnings
get_warnings(all_dt_updated, all_dd_stripped)

In [ ]:
# test get_category
get_category(all_dt_updated, all_dd_stripped)

In [ ]:
# test get_language
language = get_language(all_dt_updated, all_dd_stripped)
print(language)

In [ ]:
# test get_fandom
get_fandom(all_dt_updated, all_dd_stripped)

In [ ]:
# test get_relationships
get_relationships(all_dt_updated, all_dd_stripped)

In [ ]:
# test get_characters
get_characters(all_dt_updated, all_dd_stripped)

In [ ]:
# test get_additional_tags
get_additional_tags(all_dt_updated, all_dd_stripped)

In [ ]:
is_series(all_dd_stripped)

In [ ]:
# test get_date_published
date_published = get_date_published(all_dt_updated, all_dd_stripped)
print(date_published)
print(type(date_published))

In [ ]:
# test get_date_updated
date_updated = get_date_updated(all_dt_updated, all_dd_stripped)
print(date_updated)
print(type(date_updated))

In [ ]:
# test get_word_count
word_count = get_word_count(all_dt_updated, all_dd_stripped)
print(word_count)

In [ ]:
# test get_chapter_count
chapter_count = get_chapter_count(all_dt_updated, all_dd_stripped)
print(chapter_count)

In [ ]:
# test get_comment_count
comment_count = get_comment_count(all_dt_updated, all_dd_stripped)
print(comment_count)

In [ ]:
# test get_kudos_count
kudos_count = get_kudos_count(all_dt_updated, all_dd_stripped)
print(kudos_count)

In [ ]:
# test get_bookmarks_count
bookmarks_count = get_bookmarks_count(all_dt_updated, all_dd_stripped)
print(bookmarks_count)

In [ ]:
# test create_fanwork_dictionary
test_dict = create_fanwork_dict(all_dt_updated, all_dd_stripped)
print(test_dict)

In [ ]:
# test append_all_works_dict
append_all_works_dict(test_dict)

In [ ]:
print(len(all_works_dict))

In [ ]:
# test convert_to_json
test_json = convert_to_json(all_works_dict)
print(test_json)

In [ ]:
print(type(test_json))

In [ ]:
# test save_json_to_file
save_json_to_file(test_json)

## First 10 fanworks

In [ ]:
first_10_fanworks = fanwork_links[0:10]
print(first_10_fanworks)

In [ ]:
## first10works_dict
total_fanworks = len(first_10_fanworks)
first10works_dict = {}
for i in range(len(first_10_fanworks)):
    print("Current fanwork link:", str(i+1), "of", total_fanworks)

    ## load fanworks and create soup objects
    soup = get_html_doc_fanworks(first_10_fanworks[i])
    all_dt = get_all_dt(soup)
    all_dt_updated = update_all_dt(all_dt)
    # print(all_dt_updated)
    all_dd = get_all_dd(soup)
    print("Successfully loaded fanworks and created soup objects for link", str(i+1))

    ## clean up all_dd
    all_dd_stripped = strip_dd(all_dd)
    # print(len(all_dd_stripped))
    all_dd_stripped = cleanup_fandom_tags(all_dd_stripped)
    all_dd_stripped = cleanup_relationship_tags(all_dd_stripped)
    all_dd_stripped = cleanup_character_tags(all_dd_stripped)
    all_dd_stripped = cleanup_additional_tags(all_dd_stripped)
    print("Successfully cleaned all_dd_stripped for link", str(i+1))

    ## create individual fanwork dictionary
    fanwork_dict = create_fanwork_dict(all_dt_updated, all_dd_stripped)
    # print(fanwork_dict)
    print("Successfully created individual fanwork dictionary for link", str(i+1))

    ## append fanwork dictionary to all_works_dict1
    # first10works_dict = create_all_works_dict()
    append_all_works_dict(fanwork_dict, first10works_dict)
    print("Successfully appended individual fanwork dictionary", str(i+1), "to first10works_dict.")

In [ ]:
print(first10works_dict)

In [ ]:
print(len(first10works_dict))

In [ ]:
## convert to json and save to file
first10works_json = convert_to_json(first10works_dict)
with open('./data/first10works.json', "w") as f:
    f.write(first10works_json)

## First 20 fanworks - for G8

In [ ]:
first_20_fanworks = fanwork_links[0:20]
print(first_20_fanworks)

In [ ]:
## first20works_dict
total_fanworks = len(first_20_fanworks)
first20works_dict = {}
for i in range(len(first_20_fanworks)):
    print("Current fanwork link:", str(i+1), "of", total_fanworks)

    ## load fanworks and create soup objects
    soup = get_html_doc_fanworks(first_20_fanworks[i])
    all_dt = get_all_dt(soup)
    all_dt_updated = update_all_dt(all_dt)
    # print(all_dt_updated)
    all_dd = get_all_dd(soup)
    print("Successfully loaded fanworks and created soup objects for link", str(i+1))

    ## clean up all_dd
    all_dd_stripped = strip_dd(all_dd)
    # print(len(all_dd_stripped))
    all_dd_stripped = cleanup_fandom_tags(all_dd_stripped)
    all_dd_stripped = cleanup_relationship_tags(all_dd_stripped)
    all_dd_stripped = cleanup_character_tags(all_dd_stripped)
    all_dd_stripped = cleanup_additional_tags(all_dd_stripped)
    print("Successfully cleaned all_dd_stripped for link", str(i+1))

    ## create individual fanwork dictionary
    fanwork_dict = create_fanwork_dict(all_dt_updated, all_dd_stripped)
    # print(fanwork_dict)
    print("Successfully created individual fanwork dictionary for link", str(i+1))

    ## append fanwork dictionary to all_works_dict1
    append_all_works_dict(fanwork_dict, first20works_dict)
    print("Successfully appended individual fanwork dictionary", str(i+1), "to first20works_dict.")

In [ ]:
## convert to json and save to file
first20works_json = convert_to_json(first20works_dict)
with open('./data/first20works.json', "w") as f:
    f.write(first20works_json)

## Full fanworks list

In [ ]:
print(len(fanwork_links))

In [ ]:
fanwork_links_minus1 = fanwork_links[1:]

In [ ]:
print(len(fanwork_links_minus1))

In [ ]:
## full fanworks list
total_fanworks = len(fanwork_links)
all_candela_works_dict = {}
for i in range(len(fanwork_links)):
    print("Current fanwork link:", str(i+1), "of", total_fanworks)

    ## load fanworks and create soup objects
    soup = get_html_doc_fanworks(fanwork_links[i])
    all_dt = get_all_dt(soup)
    all_dt_updated = update_all_dt(all_dt)
    # print(all_dt_updated)
    all_dd = get_all_dd(soup)
    print("Successfully loaded fanworks and created soup objects for link", str(i+1))

    ## clean up all_dd
    all_dd_stripped = strip_dd(all_dd)
    # print(len(all_dd_stripped))
    all_dd_stripped = cleanup_fandom_tags(all_dd_stripped)
    all_dd_stripped = cleanup_relationship_tags(all_dd_stripped)
    all_dd_stripped = cleanup_character_tags(all_dd_stripped)
    all_dd_stripped = cleanup_additional_tags(all_dd_stripped)
    print("Successfully cleaned all_dd_stripped for link", str(i+1))

    ## create individual fanwork dictionary
    fanwork_dict = create_fanwork_dict(all_dt_updated, all_dd_stripped)
    # print(fanwork_dict)
    print("Successfully created individual fanwork dictionary for link", str(i+1))

    ## append fanwork dictionary to all_works_dict1
    append_all_works_dict(fanwork_dict, all_candela_works_dict)
    print("Successfully appended individual fanwork dictionary", str(i+1), "to all_candela_works_dict.")

In [ ]:
print(len(all_candela_works_dict))

In [ ]:
## convert to json and save to file
all_candela_works_json = convert_to_json(all_candela_works_dict)
with open('../../data/allcandelaworks.json', "w") as f:
    f.write(all_candela_works_json)

## Full fanworks - from individual pagination links

In [ ]:
def get_links_to_scrape(filepath):
    """Takes in a txt file of pagination links and returns a list."""
    links_to_scrape = open(filepath).readlines()
    return links_to_scrape

In [ ]:
# test get_links_to_scrape
links_to_scrape = get_links_to_scrape('./data/txt_files.txt')
print(links_to_scrape)

In [ ]:
## get pagination links to scrape
pagination_links1 = get_fanwork_links('./data/txt_files/fanwork_links1.txt')
pagination_links2 = get_fanwork_links('./data/txt_files/fanwork_links2.txt')
pagination_links3 = get_fanwork_links('./data/txt_files/fanwork_links3.txt')
pagination_links4 = get_fanwork_links('./data/txt_files/fanwork_links4.txt')
pagination_links5 = get_fanwork_links('./data/txt_files/fanwork_links5.txt')
pagination_links6 = get_fanwork_links('./data/txt_files/fanwork_links6.txt')
pagination_links7 = get_fanwork_links('./data/txt_files/fanwork_links7.txt')
pagination_links8 = get_fanwork_links('./data/txt_files/fanwork_links8.txt')
pagination_links9 = get_fanwork_links('./data/txt_files/fanwork_links9.txt')
pagination_links10 = get_fanwork_links('./data/txt_files/fanwork_links10.txt')
pagination_links11 = get_fanwork_links('./data/txt_files/fanwork_links11.txt')

In [ ]:
print(pagination_links1)
print(len(pagination_links1))

In [ ]:
pagination_links = [pagination_links1, pagination_links2, pagination_links3, pagination_links4, pagination_links5, pagination_links6, pagination_links7, pagination_links8, pagination_links9, pagination_links10, pagination_links11]

In [ ]:
print(len(pagination_links))
print(pagination_links)

In [ ]:
def remove_chapter_links(work_links):
    """Takes in a list of pared down fanwork URLs from pare_down_work_links and removes any links to individual fanwork chapters."""
    works_only_final = work_links
    for i in range(len(work_links)):
        current_link = work_links[i]
        if 'chapters' in current_link:
            works_only_final.remove(current_link)
        else:
            continue
    return works_only_final

In [ ]:
# remove chapters links from each pagination link
for i in range(len(pagination_links1)):
    current_link = pagination_links1[i]
    if 'chapters' in current_link:
        pagination_links1.remove(current_link)
    else:
        continue

In [ ]:
print(pagination_links1)
print(len(pagination_links1))

In [ ]:
print(len(pagination_links))

In [ ]:
## individual pagination links list

all_candela_works_dict1 = {}

for i in range(len(pagination_links)):
    total_fanworks = len(pagination_links[i])
    current_dict = {}
    current_pagination_link = pagination_links[i]
    print("Current pagination link: " + str(i+1) + " of", str(len(pagination_links)))
    for index in range(len(current_pagination_link)):
        print("Current fanwork link: " + str(index+1), " of", total_fanworks)

        ## load fanworks and create soup objects
        soup = get_html_doc_fanworks(fanwork_links[index])
        all_dt = get_all_dt(soup)
        all_dt_updated = update_all_dt(all_dt)
        # print(all_dt_updated)
        all_dd = get_all_dd(soup)
        print("Successfully loaded fanworks and created soup objects for link", str(index+1))

        ## clean up all_dd
        all_dd_stripped = strip_dd(all_dd)
        # print(len(all_dd_stripped))
        all_dd_stripped = cleanup_fandom_tags(all_dd_stripped)
        all_dd_stripped = cleanup_relationship_tags(all_dd_stripped)
        all_dd_stripped = cleanup_character_tags(all_dd_stripped)
        all_dd_stripped = cleanup_additional_tags(all_dd_stripped)
        print("Successfully cleaned all_dd_stripped for link", str(index+1))

        ## create individual fanwork dictionary
        fanwork_dict = create_fanwork_dict(all_dt_updated, all_dd_stripped)
        # print(fanwork_dict)
        print("Successfully created individual fanwork dictionary for link", str(index+1))

        ## append fanwork dictionary to current dictionary
        append_all_works_dict(fanwork_dict, current_dict)
        print("Successfully appended individual fanwork dictionary", str(index+1), "to current dict.")

    ## convert current dict to json and save to file
    current_json = convert_to_json(current_dict)
    json_filepath = './data/json_files/candela_page' + str(i+1) + '.json'
    with open(json_filepath, "w") as f:
        f.write(current_json)

# what am I trying to do here
# I'm trying to save each json file with a new name so they don't overwrite

In [ ]:
# code from: https://www.geeksforgeeks.org/python/how-to-merge-multiple-json-files-using-python/
def merge_json_files(file_paths, output_file):
    merged_data = []
    for path in file_paths:
        with open(path, 'r') as file:
            data = json.load(file)
            merged_data.append(data)
    with open(output_file, 'w') as outfile:
        json.dump(merged_data, outfile)

In [ ]:
# append individual jsons to all_candela_works_dict1
file_paths = ["data/json_files/candela_page1.json",
              "data/json_files/candela_page2.json",
              "data/json_files/candela_page3.json",
              "data/json_files/candela_page4.json",
              "data/json_files/candela_page5.json",
              "data/json_files/candela_page6.json",
              "data/json_files/candela_page7.json",
              "data/json_files/candela_page8.json",
              "data/json_files/candela_page9.json",
              "data/json_files/candela_page10.json",
              "data/json_files/candela_page11.json",
              ]

# create new JSON file
output_dict = {}
output_file = convert_to_json(output_dict)
output_filepath = "./data/json_files/candela_merged.json"
with open(output_filepath, 'w') as f:
    f.write(output_file)

merge_json_files(file_paths, output_filepath)
print(f"Merged data written to '{output_file}'")

In [ ]:
print(len(all_candela_works_dict1))

In [ ]:
## convert to json and save to file
all_candela_works_json1 = convert_to_json(all_candela_works_dict1)
with open('./data/allcandelaworks.json', "w") as f:
    f.write(all_candela_works_json1)

# Testing / Troubleshooting

## Round 1

In [ ]:
def check_if_rating(all_dt):
    if 'Rating' in all_dt_updated:
        print('Rating is in all_dt_updated')
    else:
        print("nope")

In [ ]:
for i in range(len(fanwork_links)):
    fanwork_links[i] = fanwork_links[i].replace("\n","")

In [ ]:
test_fanwork_list = fanwork_links[0:10]
print(test_fanwork_list)

In [ ]:
print(test_fanwork_list[0])

In [ ]:
def check_word_count(all_dt):
    if 'Word Count' in all_dt:
        print('Word Count is in all_dt')
    else:
        print("Word count is not in all_dt")

In [ ]:
# create + fill all_works_dict
total_fanworks = len(test_fanwork_list)
all_works_dict2 = {}
for i in range(len(test_fanwork_list)):
    print("Current fanwork link:", str(i+1), "of", total_fanworks)

    ## load fanworks and create soup objects
    soup = get_html_doc_fanworks(test_fanwork_list[i])
    all_dt = get_all_dt(soup)
    all_dt_updated = update_all_dt(all_dt)
    check_if_rating(all_dt_updated)
    check_word_count(all_dt_updated)
    all_dd = get_all_dd(soup)
    print("Successfully loaded fanworks and created soup objects for link", str(i+1))

    ## clean up all_dd
    text_to_strip = ['<dd>','</dd>','<ul class="commas">','/ul>','\n',' ','<li>','</li>','><aclass="tag"href="','<ddclass=','</a><']
    all_dd_stripped = strip_dd(all_dd, text_to_strip)
    dd_list1 = cleanup_rating(all_dd_stripped)
    dd_list2 = cleanup_warnings(dd_list1)
    dd_list3 = cleanup_categories(dd_list2)
    dd_list4 = multiple_tags_to_lists(dd_list3)
    indices = get_idx_multiple_tags(dd_list4)
    dd_list5 = cleanup_multiple_tags(dd_list4, indices)
    print("Successfully cleaned all_dd for link", str(i+1))

    ## create individual fanwork dictionary
    fanwork_dict = create_fanwork_dict(all_dt_updated, dd_list5)
    print("Successfully created individual fanwork dictionary for link", str(i+1))

    ## append fanwork dictionary to all_works_dict1
    append_all_works_dict(fanwork_dict, all_works_dict2)
    print("Successfully appended individual fanwork dictionary", str(i+1), "to all_works_dict2.")

In [ ]:
## convert to json and save to file
# convert to json
test_candela_works_json = convert_to_json(all_works_dict2)

# save json to file
with open('../../data/test_candela_works.json', "w") as f:
    f.write(test_candela_works_json)

## Round 2

In [ ]:
test_fanwork_list1 = fanwork_links[0:1]
print(test_fanwork_list1)
print(type(test_fanwork_list1))

In [ ]:
# create + fill all_works_dict
total_fanworks = len(test_fanwork_list1)
all_works_dict3 = {}
for i in range(len(test_fanwork_list1)):
    print("Current fanwork link:", str(i+1), "of", total_fanworks)

    ## load fanworks and create soup objects
    soup = get_html_doc_fanworks(test_fanwork_list1[i])
    all_dt = get_all_dt(soup)
    all_dt_updated = update_all_dt(all_dt)
    check_if_rating(all_dt_updated)
    print(all_dt_updated[2])
    print(type(all_dt_updated[2]))
    all_dd = get_all_dd(soup)
    print("Successfully loaded fanworks and created soup objects for link", str(i+1))

    ## clean up all_dd
    text_to_strip = ['<dd>','</dd>','<ul class="commas">','/ul>','\n',' ','<li>','</li>','><aclass="tag"href="','<ddclass=','</a><']
    all_dd_stripped = strip_dd(all_dd, text_to_strip)
    dd_list1 = cleanup_rating(all_dd_stripped)
    dd_list2 = cleanup_warnings(dd_list1)
    dd_list3 = cleanup_categories(dd_list2)
    dd_list4 = multiple_tags_to_lists(dd_list3)
    indices = get_idx_multiple_tags(dd_list4)
    dd_list5 = cleanup_multiple_tags(dd_list4, indices)
    print("Successfully cleaned all_dd for link", str(i+1))

    ## create individual fanwork dictionary
    fanwork_dict = create_fanwork_dict(all_dt_updated, dd_list5)
    print(fanwork_dict)
    print("Successfully created individual fanwork dictionary for link", str(i+1))

    ## append fanwork dictionary to all_works_dict3
    append_all_works_dict(fanwork_dict, all_works_dict3)
    print("Successfully appended individual fanwork dictionary", str(i+1), "to all_works_dict3.")

In [ ]:
## convert to json and save to file
# convert to json
test_candela_works_json1 = convert_to_json(all_works_dict3)

# save json to file
with open('../../data/test_candela_works1.json', "w") as f:
    f.write(test_candela_works_json1)

## Round 3

In [ ]:
test_fanwork_list2 = fanwork_links[1:2]
print(test_fanwork_list2)

In [ ]:
a = [10, 20, 30, 40, 50]

if 30 in a:
    print("Element exists in the list")
else:
    print("Element does not exist")

In [ ]:
# create + fill all_works_dict4
total_fanworks = len(test_fanwork_list2)
all_works_dict4 = {}
for i in range(len(test_fanwork_list2)):
    print("Current fanwork link:", str(i+1), "of", total_fanworks)

    ## load fanworks and create soup objects
    soup = get_html_doc_fanworks(test_fanwork_list2[i])
    all_dt = get_all_dt(soup)
    all_dt_updated = update_all_dt(all_dt)
    check_if_rating(all_dt_updated)
    all_dd = get_all_dd(soup)
    print("Successfully loaded fanworks and created soup objects for link", str(i+1))

    ## clean up all_dd
    text_to_strip = ['<dd>','</dd>','<ul class="commas">','/ul>','\n',' ','<li>','</li>','><aclass="tag"href="','<ddclass=','</a><']
    all_dd_stripped = strip_dd(all_dd, text_to_strip)
    dd_list1 = cleanup_rating(all_dd_stripped)
    dd_list2 = cleanup_warnings(dd_list1)
    dd_list3 = cleanup_categories(dd_list2)
    dd_list4 = multiple_tags_to_lists(dd_list3)
    indices = get_idx_multiple_tags(dd_list4)
    dd_list5 = cleanup_multiple_tags(dd_list4, indices)
    print("Successfully cleaned all_dd for link", str(i+1))

    ## create individual fanwork dictionary
    fanwork_dict = create_fanwork_dict(all_dt_updated, dd_list5)
    print("Successfully created individual fanwork dictionary for link", str(i+1))

    ## append fanwork dictionary to all_works_dict4
    append_all_works_dict(fanwork_dict, all_works_dict4)
    print("Successfully appended individual fanwork dictionary", str(i+1), "to all_works_dict4.")

In [ ]:
## convert to json and save to file
# convert to json
test_candela_works_json2 = convert_to_json(all_works_dict4)

# save json to file
with open('../../data/test_candela_works2.json', "w") as f:
    f.write(test_candela_works_json2)